## **AI And Biotechnology/Bioinformatics**

## **AI and Drug Discovery Course: QSAR Modeling**
This notebook demonstrates how to collect and preprocess bioactivity data from ChEMBL for QSAR modeling

# **Part 3: Descriptor Calculation**

PaDELPy is a Python wrapper for the PaDEL-Descriptor (molecular descriptor calculation) software.  

It provide the following descriptors/fingerprint:  
* 2D Descriptors: Molecular weight, atom counts, ring counts, and bond types
* 3D Descriptors : Molecular volume and spatial surface area
* Fingerprints: Binary bit vectors (0s and 1s) representing the presence or absence of specific chemical fragments (like PubChem)

$$\begin{bmatrix} 1 & 0 & 1 & 1 & 0 & 0 \end{bmatrix}$$

## **Install PaDELpy**

In [ ]:
!pip install padelpy

## **Import libraries**

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from padelpy import padeldescriptor

## **Load dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('df_lipinski.csv')
df.head()

In [ ]:
data = df[['canonical_smiles', 'molecule_chembl_id']]
data.head()

## **Convert to .smi format**

In [ ]:
df_smi = data['canonical_smiles'].to_csv('smiles_chembl.smi', index=None, header=None)

In [ ]:
! cat smiles_chembl.smi | head

## **Calculate molecular Pubchem Fingerprints using "padeldescriptor" function**


In [ ]:
padeldescriptor(mol_dir= "smiles_chembl.smi",
                d_file='pubchem_fingerprints.csv',
                fingerprints = True,
                retainorder= True,
                #removesalt = True, standardizetautomers = True, standardizenitro=True
                )

In [ ]:
!ls -lh pubchem_fingerprints.csv

In [ ]:
df_fingerprint = pd.read_csv("pubchem_fingerprints.csv")
df_fingerprint.head()

## **Prepare Dataset for ML**

In [ ]:
df.head()

In [ ]:
# Select only the columns we need for ML
meta_cols = df[['molecule_chembl_id', 'bioactivity_class', 'pIC50']]

# Reset index to ensure proper alignment
meta_cols = meta_cols.reset_index(drop=True)
df_fingerprint = df_fingerprint.reset_index(drop=True)

# Combine meta data with fingerprints
combined_df = pd.concat([meta_cols, df_fingerprint.drop(df_fingerprint.columns[0], axis=1)], axis=1)

# Inspect the first few rows
combined_df.head()


## **Save and download the dataset**

In [ ]:
# Save as CSV
combined_df.to_csv("QSAR_dataset.csv", index=False)
print("Combined dataset saved as QSAR_dataset.csv")

# Download file in Colab
files.download("QSAR_dataset.csv")

# **Calculate other fingerprints**

## **Download xml Files from Github**

In [ ]:
!wget https://github.com/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/raw/main/padel_descriptors_xml.zip

## **Unzip all files**

In [ ]:
!unzip padel_descriptors_xml.zip

**PubchemFingerprinter**

Primary Use: Virtual screening of massive chemical databases and general-purpose bioactivity classification

**MACCSFingerprinter**

Primary Use: Rapid filtering during early-stage virtual screening and similarity searching. Because it is small (only 166 bits), it is computationally incredibly fast

**SubstructureFingerprinter & SubstructureFingerprintCount**

Primary Use: Predicting metabolic soft spots, structural toxicology (toxicity profiling), or specific off-target interactions.

**ExtendedFingerprinter**

Primary Use: Modeling highly cyclic compounds, natural products, or complex macrocycles.

**GraphOnlyFingerprinter**

Primary Use: Scaffold Hopping. This is a powerful technique where you want to find a brand-new chemical compound that treats the same disease but looks entirely different to avoid patent issues

**KlekotaRothFingerprinter & KlekotaRothFingerprintCount**

Primary Use: Deep learning models, complex bioactivity mapping, and advanced phenotypic screening.

**AtomPairs2DFingerprinter & AtomPairs2DFingerprintCount**

Primary Use: Pharmacophore Modeling and predicting binding site matching

**EStateFingerprinterPrimary Use:**
 Quantitative Structure-Property Relationship (QSPR) modeling, solubility ($\text{LogS}$) prediction, and pharmacokinetics ($\text{ADME}$) calculations

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from google.colab import files

# 1. Map out all 12 unzipped XML fingerprinter parameters
fingerprint_data = {
    "fingerprint": [
        "AtomPairs2DFingerprintCount", "AtomPairs2DFingerprinter",
        "EStateFingerprinter", "ExtendedFingerprinter", "Fingerprinter",
        "GraphOnlyFingerprinter", "KlekotaRothFingerprintCount",
        "KlekotaRothFingerprinter", "MACCSFingerprinter", "PubchemFingerprinter",
        "SubstructureFingerprintCount", "SubstructureFingerprinter"
    ],
    "abbreviation": [
        "AP2DCount", "AP2D", "EState", "ExtFP", "StandardFP",
        "GraphFP", "KRCount", "KR", "MACCS", "PubChem", "SubCount", "SubFP"
    ],
    "number(bits)": [
        780, 780, 79, 1024, 1024, 1024, 4860, 4860, 166, 881, 307, 307
    ],
    "fingerprint pattern type": [
        "Atom Pairs (Frequency Count)", "Atom Pairs (Binary Presence)",
        "Electrotopological State Intrinsic Values", "Circular / Topological Pathways (Extended)",
        "Standard Path-based / Linear Fragments", "Path-based (Ignoring Bond Orders/Aromaticity)",
        "Chemical Substructure Keys (Frequency Count)", "Chemical Substructure Keys (Binary Presence)",
        "Predefined Structural Keys (MDL Keys)", "Predefined Structural Keys (CIDs/Substructures)",
        "Functional Group Keys (Frequency Count)", "Functional Group Keys (Binary Presence)"
    ],
    "description": [
        "Counts the frequency of pairs of atoms separated by a defined topological distance.",
        "Binary 1/0 representation of whether specific atom pairs exist at given bond distances.",
        "Calculates electronic and structural attributes of atoms based on valence states.",
        "Extends standard linear path fingerprints to account for broader structural environments.",
        "Analyzes continuous linear chains and rings of atoms up to a specified length.",
        "Maintains structural connectivity sequences while ignoring formal bond types.",
        "Counts how many times specific complex chemical patterns/rings occur in a structure.",
        "Binary 1/0 tracking for the presence of complex structural motifs curated by Klekota and Roth.",
        "Standard 166-bit checklist evaluating specific foundational functional groups.",
        "Standard 881-bit checklist from NCBI evaluating elements, rings, and atoms.",
        "Counts the occurrences of 307 specific functional substructures.",
        "Binary 1/0 checklist tracking 307 predefined standard functional groups."
    ]
}

# Convert to DataFrame
df_fp = pd.DataFrame(fingerprint_data)

# 2. Setup professional OpenPyXL Workbook
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Fingerprint Reference Guide"
ws.views.sheetView[0].showGridLines = True  # Ensure grid lines stay visible

# Append headers and all rows
headers = list(df_fp.columns)
ws.append(headers)
for row in df_fp.values.tolist():
    ws.append(row)

# 3. Apply Professional Styling (Deep Navy & Accent Blue Theme)
navy_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
white_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
zebra_fill = PatternFill(start_color="F2F6FA", end_color="F2F6FA", fill_type="solid")
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
)

# Style Header Row
ws.row_dimensions[1].height = 26
for col_idx in range(1, len(headers) + 1):
    cell = ws.cell(row=1, column=col_idx)
    cell.fill = navy_fill
    cell.font = white_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

# Style Data Rows (Zebra striping, numeric formats, auto text-wrap)
for row_idx in range(2, ws.max_row + 1):
    ws.row_dimensions[row_idx].height = 22
    is_even = (row_idx % 2 == 0)
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=row_idx, column=col_idx)
        cell.border = thin_border
        if is_even:
            cell.fill = zebra_fill

        header_name = headers[col_idx-1]
        if header_name == "number(bits)":
            cell.number_format = '#,##0'
            cell.alignment = Alignment(horizontal='right', vertical='center')
        else:
            cell.alignment = Alignment(horizontal='left', vertical='center', wrap_text=(header_name == "description"))

# 4. Set tailored column layout padding
column_widths = {"A": 32, "B": 15, "C": 15, "D": 42, "E": 65}
for col_letter, width in column_widths.items():
    ws.column_dimensions[col_letter].width = width

# 5. Save and download directly from Google Colab
filename = "padel_fingerprints_dictionary.xlsx"
wb.save(filename)
files.download(filename)

In [ ]:
# Specify the XML file for SubstructureFingerprinter directly
Substruc_fp = "SubstructureFingerprinter.xml"

# Calculate Substructure fingerprints
padeldescriptor(
    mol_dir='smiles_chembl.smi',
    d_file='Substructure_fingerprints.csv',
    fingerprints=True,
    descriptortypes= Substruc_fp,
    retainorder=True
    # removesalt=True, standardizetautomers=True
)

df_fp = pd.read_csv('Substructure_fingerprints.csv')

df_fp.to_excel('Substructure_fingerprints.xlsx', index=False)

files.download('Substructure_fingerprints.xlsx')

**Graphs of fingerprints**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# 1. Load the PubChem fingerprint dataset generated by PaDEL
csv_file = "pubchem_fingerprints.csv"

try:
    df_fingerprint = pd.read_csv(csv_file)
    print(f"Successfully loaded '{csv_file}'!")
    print(f"Dataset dimensions: {df_fingerprint.shape[0]} compounds and {df_fingerprint.shape[1]} columns.\n")
except FileNotFoundError:
    print(f"❌ Error: '{csv_file}' not found in your current directory.")
    print("Please make sure you have run the pd.read_csv step or that the file exists in your left sidebar file explorer.")

# 2. Extract only the binary bit columns (drop the row tracking identifiers)
# This removes the 'Name' column containing 'AUTOGEN_smiles_chembl_...'
bit_columns = df_fingerprint.drop(columns=['Name', 'Identifier'], errors='ignore')

# Set professional plotting layout theme
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# ==============================================================================
# GRAPH 1: Feature Density Distribution (Active Bits Per Compound)
# ==============================================================================
# Sum horizontally across columns to see how many fragments are present per molecule
active_bits_per_molecule = bit_columns.sum(axis=1)

plt.figure(figsize=(9, 5.5))
sns.histplot(
    active_bits_per_molecule,
    kde=True,
    color='#1F4E79',       # Professional deep navy blue
    bins=20,
    line_kws={'linewidth': 2.5},
    alpha=0.8
)

plt.title('Structural Feature Density (PubChem Fingerprints)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Active Bits (Presence of Core Fragments / Substructures)', fontsize=12)
plt.ylabel('Frequency (Count of Compounds)', fontsize=12)
plt.tight_layout()

# Save high-resolution version to your Colab workspace folder
plt.savefig('pubchem_bit_density_distribution.png', dpi=300)
plt.show()

print("📊 Graph 1 generated: 'pubchem_bit_density_distribution.png' saved.")


# ==============================================================================
# GRAPH 2: Top 10 Most Prevalent Specific Chemical Bits in Your Library
# ==============================================================================
# Sum vertically down rows to find which fragments are most common across all compounds
top_fragments = bit_columns.sum(axis=0).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5.5))
# Using a clean scientific gradient color palette (Crest)
sns.barplot(
    x=top_fragments.values,
    y=top_fragments.index,
    palette='crest',
    alpha=0.9
)

plt.title('Top 10 Most Prevalent Specific PubChem Bits in Dataset', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Molecules Containing the Feature', fontsize=12)
plt.ylabel('PubChem Fingerprint Bit ID', fontsize=12)
plt.tight_layout()

# Save high-resolution version to your Colab workspace folder
plt.savefig('pubchem_top_prevalent_bits.png', dpi=300)
plt.show()

print("📊 Graph 2 generated: 'pubchem_top_prevalent_bits.png' saved.")
print("\n🎉 Analysis Complete! Check your left sidebar folder to download the high-res PNGs.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Re-create df_fp with fingerprint metadata to ensure the correct DataFrame is used
# This assumes 'fingerprint_data' dictionary is available from a previously executed cell.
fingerprint_data = {
    "fingerprint": [
        "AtomPairs2DFingerprintCount", "AtomPairs2DFingerprinter",
        "EStateFingerprinter", "ExtendedFingerprinter", "Fingerprinter",
        "GraphOnlyFingerprinter", "KlekotaRothFingerprintCount",
        "KlekotaRothFingerprinter", "MACCSFingerprinter", "PubchemFingerprinter",
        "SubstructureFingerprintCount", "SubstructureFingerprinter"
    ],
    "abbreviation": [
        "AP2DCount", "AP2D", "EState", "ExtFP", "StandardFP",
        "GraphFP", "KRCount", "KR", "MACCS", "PubChem", "SubCount", "SubFP"
    ],
    "number(bits)": [
        780, 780, 79, 1024, 1024, 1024, 4860, 4860, 166, 881, 307, 307
    ],
    "fingerprint pattern type": [
        "Atom Pairs (Frequency Count)", "Atom Pairs (Binary Presence)",
        "Electrotopological State Intrinsic Values", "Circular / Topological Pathways (Extended)",
        "Standard Path-based / Linear Fragments", "Path-based (Ignoring Bond Orders/Aromaticity)",
        "Chemical Substructure Keys (Frequency Count)", "Chemical Substructure Keys (Binary Presence)",
        "Predefined Structural Keys (MDL Keys)", "Predefined Structural Keys (CIDs/Substructures)",
        "Functional Group Keys (Frequency Count)", "Functional Group Keys (Binary Presence)"
    ],
    "description": [
        "Counts the frequency of pairs of atoms separated by a defined topological distance.",
        "Binary 1/0 representation of whether specific atom pairs exist at given bond distances.",
        "Calculates electronic and structural attributes of atoms based on valence states.",
        "Extends standard linear path fingerprints to account for broader structural environments.",
        "Analyzes continuous linear chains and rings of atoms up to a specified length.",
        "Maintains structural connectivity sequences while ignoring formal bond types.",
        "Counts how many times specific complex chemical patterns/rings occur in a structure.",
        "Binary 1/0 tracking for the presence of complex structural motifs curated by Klekota and Roth.",
        "Standard 166-bit checklist evaluating specific foundational functional groups.",
        "Standard 881-bit checklist from NCBI evaluating elements, rings, and atoms.",
        "Counts the occurrences of 307 specific functional substructures.",
        "Binary 1/0 checklist tracking 307 predefined standard functional groups."
    ]
}

df_fp = pd.DataFrame(fingerprint_data)

# Set style
sns.set_theme(style="whitegrid")

# Create plot
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_fp,
    x='number(bits)',
    y='abbreviation',
    hue='fingerprint pattern type',
    dodge=False,
    palette='viridis'
)

plt.title('Comparison of PaDEL Fingerprint Dimensionality', fontsize=14, fontweight='bold')
plt.xlabel('Number of Bits')
plt.ylabel('Fingerprint Method')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Pattern Type")
plt.tight_layout()
plt.show()